# Quotely — Évaluation automatique du système de citation

**Pipeline :**
1. Récupère les articles indexés dans Quotely
2. Pour chaque article, demande à Mistral (LM Studio) d'écrire un passage académique de ~400 tokens sur son contenu
3. Envoie ce passage à `/suggest` et vérifie si l'article cible est retrouvé
4. Calcule les métriques : Hit@k, MRR, Precision@k
5. Sauvegarde tout (checkpointing) + génère les graphiques

**Checkpointing** : les résultats sont sauvegardés ligne par ligne dans `results/raw.jsonl`. Si le notebook plante, relance-le — il reprend là où il s'est arrêté.

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
BACKEND_URL     = "http://127.0.0.1:7331"
LM_STUDIO_URL   = "http://10.3.160.196:1234/v1/chat/completions"
LM_MODEL        = "mistralai/ministral-3-14b-reasoning"

N_PAPERS        = 100   # max articles à évaluer (None = tous)
N_SUGGESTIONS   = 15    # nombre de suggestions Quotely à récupérer
PASSAGE_TOKENS  = 420   # tokens max pour le passage Mistral
TEMPERATURE     = 0.7
TIMEOUT_LLM     = 120   # secondes
TIMEOUT_BACKEND = 30

RESULTS_DIR     = "results"
RAW_FILE        = f"{RESULTS_DIR}/raw.jsonl"       # une ligne JSON par article
SUMMARY_FILE    = f"{RESULTS_DIR}/summary.json"    # métriques agrégées
PASSAGES_FILE   = f"{RESULTS_DIR}/passages.jsonl"  # passages générés (réutilisables)

In [ ]:
import os, json, time, random, datetime
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from pathlib import Path

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Dossier résultats : {Path(RESULTS_DIR).resolve()}")

## 1. Connexion aux services

In [ ]:
# ─── Vérifie Quotely ──────────────────────────────────────────────────────────
r = requests.get(f"{BACKEND_URL}/health", timeout=5)
health = r.json()
print(f"✅ Quotely: {health['indexed_chunks']} chunks indexés")

# ─── Vérifie LM Studio ───────────────────────────────────────────────────────
try:
    r2 = requests.post(
        LM_STUDIO_URL,
        json={"model": LM_MODEL, "messages": [{"role": "user", "content": "ping"}], "max_tokens": 5},
        timeout=10
    )
    print(f"✅ LM Studio: {LM_MODEL} opérationnel")
except Exception as e:
    print(f"❌ LM Studio inaccessible : {e}")

## 2. Chargement des articles

In [ ]:

import re as _re2

papers = requests.get(f"{BACKEND_URL}/papers", timeout=10).json()
print(f"{len(papers)} articles disponibles")

# Extensions de fichiers connues → titre qui est en fait un nom de fichier
_FILE_EXT_RE = _re2.compile(r'\.(pdf|rtf|qxd|docx|doc|pptx|xlsx|md|txt|odt|png|jpg)\b', _re2.IGNORECASE)
_GENERIC_TITLES = {
    "1", "2", "3", "sommaire", "introduction", "conclusion",
    "untitled", "sans titre", "titre à déterminer", "titre",
    "legend", "légende photo :", "avertissement",
}

def _is_valid_title(t: str) -> bool:
    if not t or len(t.strip()) < 15:
        return False
    tl = t.strip().lower()
    if tl in _GENERIC_TITLES:
        return False
    if _FILE_EXT_RE.search(t):
        return False
    return True

papers = [p for p in papers if _is_valid_title(p.get('title', ''))]
print(f"{len(papers)} articles après filtrage des titres invalides")

# Sélection aléatoire reproductible
random.seed(42)
if N_PAPERS and len(papers) > N_PAPERS:
    eval_papers = random.sample(papers, N_PAPERS)
else:
    eval_papers = papers

print(f"→ {len(eval_papers)} articles sélectionnés pour l'évaluation")

# Affiche un aperçu
pd.DataFrame(eval_papers)[['bibtex_key','title','authors','year']].head(10)


## 3. Fonctions utilitaires

In [ ]:

import re as _re

def get_paper_chunks(bibtex_key: str, title: str, n: int = 4) -> list[str]:
    """Récupère les vrais chunks stockés pour ce papier via /chunks."""
    try:
        r = requests.get(
            f"{BACKEND_URL}/chunks",
            params={"key": bibtex_key, "n": n},
            timeout=TIMEOUT_BACKEND
        )
        return r.json().get("chunks", [])
    except Exception:
        return []


def _strip_reasoning(text: str) -> str:
    """Supprime les balises de raisonnement des modèles reasoning (Mistral, DeepSeek…)."""
    text = _re.sub(r"\[THINK\].*?\[/THINK\]", "", text, flags=_re.DOTALL | _re.IGNORECASE)
    text = _re.sub(r"<think>.*?</think>",       "", text, flags=_re.DOTALL | _re.IGNORECASE)
    text = _re.sub(r"<\|thinking\|>.*?<\|/thinking\|>", "", text, flags=_re.DOTALL)
    return text.strip()


SYSTEM_PROMPT = """Tu es un chercheur académique expert. Ta tâche est d'écrire un passage de ~400 tokens 
dans le style d'un article scientifique sur le sujet d'un document donné.
RÈGLES STRICTES :
- N'écris PAS le titre exact du document, ni les noms des auteurs
- Décris les concepts clés, la problématique, les apports ou la méthodologie
- Écris dans la même langue que le document (français si le titre est en français)
- Style académique, dense, factuel — comme un extrait de thèse ou d'article de revue
- Longueur : exactement ~400 tokens, pas de titre, pas de conclusion
- Réponds UNIQUEMENT avec le passage, sans méta-commentaire"""


def generate_passage(paper: dict, chunks: list[str]) -> dict:
    """Appelle Mistral pour générer un passage académique sur le papier."""
    context = "\n".join(f"- {c[:400]}" for c in chunks[:3]) if chunks else "(pas d'extrait disponible)"

    user_prompt = f"""Document à paraphraser :
Titre : {paper['title']}
Auteur(s) : {paper['authors'] or 'Inconnu'}
Année : {paper['year']}

Extraits du document :
{context}

Écris maintenant le passage académique de ~400 tokens sur ce sujet."""

    payload = {
        "model": LM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        "max_tokens": PASSAGE_TOKENS,
        "temperature": TEMPERATURE,
        "stream": False
    }

    t0 = time.time()
    r = requests.post(LM_STUDIO_URL, json=payload, timeout=TIMEOUT_LLM)
    elapsed = round(time.time() - t0, 2)

    resp = r.json()
    raw_passage  = resp["choices"][0]["message"]["content"]
    passage      = _strip_reasoning(raw_passage)
    tokens_used  = resp.get("usage", {}).get("completion_tokens", 0)

    return {"passage": passage, "tokens": tokens_used, "latency_s": elapsed}


def query_quotely(passage: str, n: int = N_SUGGESTIONS) -> list[dict]:
    """Envoie le passage à Quotely et retourne les suggestions."""
    r = requests.post(
        f"{BACKEND_URL}/suggest",
        json={"text": passage, "n": n},
        timeout=TIMEOUT_BACKEND
    )
    return r.json().get("citations", [])


def compute_rank(target_key: str, suggestions: list[dict]) -> int | None:
    """Retourne le rang (1-indexé) de target_key dans les suggestions, ou None."""
    for i, s in enumerate(suggestions):
        if s["bibtex_key"] == target_key:
            return i + 1
    return None


print("✅ Fonctions définies")


## 4. Checkpointing — reprise si interruption

In [ ]:
# Charge les résultats déjà calculés
already_done = set()
if Path(RAW_FILE).exists():
    with open(RAW_FILE) as f:
        for line in f:
            row = json.loads(line)
            already_done.add(row["bibtex_key"])
    print(f"♻️  Reprise : {len(already_done)} articles déjà évalués")
else:
    print("🆕 Démarrage à zéro")

remaining = [p for p in eval_papers if p["bibtex_key"] not in already_done]
print(f"→ {len(remaining)} articles restants à évaluer")

## 5. Boucle d'évaluation principale

In [ ]:
errors = []

for i, paper in enumerate(remaining):
    key   = paper["bibtex_key"]
    title = paper["title"][:80]
    print(f"[{i+1:3d}/{len(remaining)}] {key[:35]:35s} | {title[:50]}")

    row = {
        "bibtex_key":   key,
        "title":        paper["title"],
        "authors":      paper["authors"],
        "year":         paper["year"],
        "file_path":    paper.get("file_path", ""),
        "timestamp":    datetime.datetime.now().isoformat(),
        # champs remplis plus bas
        "passage":            None,
        "passage_tokens":     None,
        "llm_latency_s":      None,
        "suggestions":        [],
        "rank":               None,    # rang de cet article dans les résultats
        "score_at_rank":      None,    # score Quotely à ce rang
        "hit_at_1":           False,
        "hit_at_3":           False,
        "hit_at_5":           False,
        "hit_at_10":          False,
        "reciprocal_rank":    0.0,
        "error":              None,
        "chunks_used":        [],
    }

    try:
        # ── Étape 1 : extraits réels du papier ────────────────────────────────
        chunks = get_paper_chunks(key, paper["title"])
        row["chunks_used"] = chunks

        # ── Étape 2 : génération de passage par Mistral ───────────────────────
        gen = generate_passage(paper, chunks)
        row["passage"]        = gen["passage"]
        row["passage_tokens"] = gen["tokens"]
        row["llm_latency_s"]  = gen["latency_s"]

        # ── Étape 3 : requête Quotely ─────────────────────────────────────────
        suggestions = query_quotely(gen["passage"])
        row["suggestions"] = [
            {"bibtex_key": s["bibtex_key"], "title": s["title"], "score": s["score"]}
            for s in suggestions
        ]

        # ── Étape 4 : calcul du rang ──────────────────────────────────────────
        rank = compute_rank(key, suggestions)
        row["rank"] = rank

        if rank is not None:
            row["score_at_rank"]   = suggestions[rank-1]["score"]
            row["hit_at_1"]        = rank <= 1
            row["hit_at_3"]        = rank <= 3
            row["hit_at_5"]        = rank <= 5
            row["hit_at_10"]       = rank <= 10
            row["reciprocal_rank"] = round(1 / rank, 4)
            status = f"✅ rang {rank}  score={row['score_at_rank']:.2f}"
        else:
            status = f"❌ non trouvé (top {N_SUGGESTIONS})"

        print(f"         → {status}")

    except Exception as e:
        row["error"] = str(e)
        errors.append({"key": key, "error": str(e)})
        print(f"         ⚠️  ERREUR: {e}")

    # ── Sauvegarde immédiate (checkpoint) ─────────────────────────────────────
    with open(RAW_FILE, "a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Sauvegarde aussi le passage séparément pour réutilisation
    if row["passage"]:
        with open(PASSAGES_FILE, "a") as f:
            f.write(json.dumps({
                "bibtex_key": key, "passage": row["passage"]
            }, ensure_ascii=False) + "\n")


print(f"\n✅ Évaluation terminée. {len(errors)} erreurs.")
if errors:
    print("Erreurs:", errors)

## 6. Calcul des métriques

In [ ]:
# Charge tous les résultats
records = []
with open(RAW_FILE) as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
df_valid = df[df["error"].isna()].copy()

print(f"Total évalué : {len(df)}")
print(f"Sans erreur  : {len(df_valid)}")
print(f"Avec erreur  : {len(df) - len(df_valid)}")
df_valid.head(5)

In [ ]:
n = len(df_valid)

metrics = {
    "n_evaluated":            n,
    "n_errors":               int(df["error"].notna().sum()),
    "hit_at_1":               round(df_valid["hit_at_1"].mean(), 4),
    "hit_at_3":               round(df_valid["hit_at_3"].mean(), 4),
    "hit_at_5":               round(df_valid["hit_at_5"].mean(), 4),
    "hit_at_10":              round(df_valid["hit_at_10"].mean(), 4),
    "mrr":                    round(df_valid["reciprocal_rank"].mean(), 4),
    "not_found_rate":         round((df_valid["rank"].isna()).mean(), 4),
    "mean_rank_when_found":   round(df_valid[df_valid["rank"].notna()]["rank"].mean(), 2),
    "median_rank_when_found": float(df_valid[df_valid["rank"].notna()]["rank"].median()),
    "mean_score_when_found":  round(df_valid[df_valid["score_at_rank"].notna()]["score_at_rank"].mean(), 4),
    "mean_llm_latency_s":     round(df_valid["llm_latency_s"].mean(), 2),
    "total_passage_tokens":   int(df_valid["passage_tokens"].sum()),
    "timestamp":              datetime.datetime.now().isoformat(),
}

# Sauvegarde du résumé
with open(SUMMARY_FILE, "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

# Affichage
print("\n" + "═"*50)
print("         MÉTRIQUES QUOTELY EVAL")
print("═"*50)
print(f"  Articles évalués   : {metrics['n_evaluated']}")
print(f"  Non trouvé         : {metrics['not_found_rate']*100:.1f}%")
print("")
print(f"  Hit@1              : {metrics['hit_at_1']*100:.1f}%")
print(f"  Hit@3              : {metrics['hit_at_3']*100:.1f}%")
print(f"  Hit@5              : {metrics['hit_at_5']*100:.1f}%")
print(f"  Hit@10             : {metrics['hit_at_10']*100:.1f}%")
print(f"  MRR                : {metrics['mrr']:.4f}")
print(f"  Rang moyen (trouvé): {metrics['mean_rank_when_found']:.1f}")
print(f"  Score moyen        : {metrics['mean_score_when_found']:.3f}")
print("═"*50)

## 7. Visualisations

In [ ]:
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
COLORS = {"hit": "#2ecc71", "miss": "#e74c3c", "accent": "#3498db", "bg": "#1e1e2e"}
plt.rcParams.update({"figure.dpi": 150, "font.family": "sans-serif", "axes.spines.top": False, "axes.spines.right": False})

# ── Figure 1 : Hit@k bar chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Quotely — Évaluation automatique", fontsize=15, fontweight="bold", y=1.02)

ax = axes[0]
ks    = [1, 3, 5, 10]
hits  = [metrics[f"hit_at_{k}"] * 100 for k in ks]
bars  = ax.bar([f"Hit@{k}" for k in ks], hits, color=[COLORS["hit"] if h >= 50 else COLORS["accent"] for h in hits], edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, hits):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8, f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_ylim(0, 105)
ax.set_ylabel("% articles retrouvés")
ax.set_title(f"Taux de rappel par rang (n={n})")
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

# ── Figure 2 : Distribution des rangs ────────────────────────────────────────
ax2 = axes[1]
found_ranks = df_valid[df_valid["rank"].notna()]["rank"].astype(int)
not_found   = df_valid["rank"].isna().sum()

rank_counts = found_ranks.value_counts().sort_index()
all_ranks = list(range(1, N_SUGGESTIONS + 1))
counts    = [rank_counts.get(r, 0) for r in all_ranks]
bar_colors = [COLORS["hit"] if r <= 5 else COLORS["accent"] if r <= 10 else "#95a5a6" for r in all_ranks]
ax2.bar(all_ranks, counts, color=bar_colors, edgecolor="white", linewidth=0.5)
miss_patch  = mpatches.Patch(color=COLORS["hit"],   label="Top 5")
mid_patch   = mpatches.Patch(color=COLORS["accent"], label="Top 6-10")
low_patch   = mpatches.Patch(color="#95a5a6",        label="> 10")
ax2.legend(handles=[miss_patch, mid_patch, low_patch], fontsize=9)
ax2.set_xlabel("Rang de l'article cible")
ax2.set_ylabel("Nombre d'articles")
ax2.set_title(f"Distribution des rangs (non trouvés: {not_found})")
ax2.set_xticks(all_ranks)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/hit_at_k.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Sauvegardé: {RESULTS_DIR}/figures/hit_at_k.png")

In [ ]:
# ── Figure 3 : Score Quotely vs rang ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
df_found = df_valid[df_valid["rank"].notna()].copy()
df_found["rank"] = df_found["rank"].astype(int)

scatter = ax.scatter(
    df_found["rank"],
    df_found["score_at_rank"],
    c=df_found["rank"],
    cmap="RdYlGn_r",
    alpha=0.7, s=60, edgecolors="white", linewidths=0.3
)
plt.colorbar(scatter, ax=ax, label="Rang")
ax.set_xlabel("Rang")
ax.set_ylabel("Score Quotely")
ax.set_title("Score de confiance vs rang de l'article cible")

# ── Figure 4 : Camembert Trouvé / Non trouvé ─────────────────────────────────
ax2 = axes[1]
found_top5  = int(df_valid["hit_at_5"].sum())
found_top10 = int(df_valid["hit_at_10"].sum()) - found_top5
found_rest  = int(df_valid["rank"].notna().sum()) - found_top5 - found_top10
not_found   = int(df_valid["rank"].isna().sum())

sizes  = [found_top5, found_top10, found_rest, not_found]
labels = [f"Top 5 ({found_top5})", f"Top 6-10 ({found_top10})",
          f"Top 11-15 ({found_rest})", f"Non trouvé ({not_found})"]
colors = [COLORS["hit"], COLORS["accent"], "#f39c12", COLORS["miss"]]
wedges, _, autotexts = ax2.pie(
    sizes, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=90, pctdistance=0.75,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title(f"Répartition sur {n} articles")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/score_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Sauvegardé: {RESULTS_DIR}/figures/score_distribution.png")

In [ ]:
# ── Figure 5 : Top 20 cas les mieux et les moins bien retrouvés ───────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Meilleurs : rang 1 avec score le plus élevé
top_good = df_valid[df_valid["hit_at_1"] == True].nlargest(15, "score_at_rank")[["bibtex_key", "score_at_rank"]]
axes[0].barh(top_good["bibtex_key"].str[:30], top_good["score_at_rank"], color=COLORS["hit"], edgecolor="white")
axes[0].set_title("Top hits — Score le plus élevé au rang 1")
axes[0].set_xlabel("Score Quotely")
axes[0].invert_yaxis()

# Pires : non trouvés ou rang élevé
df_worst = df_valid.copy()
df_worst["rank_eff"] = df_worst["rank"].fillna(N_SUGGESTIONS + 1)
top_bad = df_worst.nlargest(15, "rank_eff")[["bibtex_key", "rank_eff"]]
colors_bad = [COLORS["miss"] if r > N_SUGGESTIONS else "#e67e22" for r in top_bad["rank_eff"]]
axes[1].barh(top_bad["bibtex_key"].str[:30], top_bad["rank_eff"], color=colors_bad, edgecolor="white")
axes[1].axvline(N_SUGGESTIONS, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("Pires cas — Rang effectif le plus élevé")
axes[1].set_xlabel(f"Rang (>{N_SUGGESTIONS} = non trouvé)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/best_worst.png", bbox_inches="tight", dpi=150)
plt.show()

## 8. Analyse qualitative — exemples de passages

In [ ]:
# ── Meilleur exemple : rank 1, score max ──────────────────────────────────────
best = df_valid[df_valid["hit_at_1"]].nlargest(1, "score_at_rank").iloc[0]
print("=" * 70)
print(f"✅ MEILLEUR EXEMPLE — rang 1, score {best['score_at_rank']:.3f}")
print(f"   Cible : {best['title']}")
print(f"   Clé   : {best['bibtex_key']}")
print("\nPassage généré par Mistral :")
print("-" * 70)
print(best["passage"][:800])
print("\nSuggestions Quotely :")
for j, s in enumerate(best["suggestions"][:5]):
    marker = "← ✅" if s["bibtex_key"] == best["bibtex_key"] else ""
    print(f"  [{j+1}] {s['bibtex_key']:35s} score={s['score']:.3f} {marker}")

In [ ]:
# ── Pire exemple : non trouvé ─────────────────────────────────────────────────
not_found_df = df_valid[df_valid["rank"].isna()]
if len(not_found_df) > 0:
    worst = not_found_df.iloc[0]
    print("=" * 70)
    print(f"❌ EXEMPLE NON TROUVÉ")
    print(f"   Cible : {worst['title']}")
    print(f"   Clé   : {worst['bibtex_key']}")
    print("\nPassage généré par Mistral :")
    print("-" * 70)
    print(str(worst["passage"])[:800])
    print("\nSuggestions Quotely (ce qu'il a proposé à la place) :")
    for s in worst.get("suggestions", [])[:5]:
        print(f"  {s['bibtex_key']:35s} score={s['score']:.3f}")

## 9. Export CSV complet

In [ ]:
csv_path = f"{RESULTS_DIR}/evaluation_results.csv"
export_cols = ["bibtex_key", "title", "authors", "year",
               "rank", "score_at_rank", "hit_at_1", "hit_at_3",
               "hit_at_5", "hit_at_10", "reciprocal_rank",
               "passage_tokens", "llm_latency_s", "error"]
df_valid[export_cols].to_csv(csv_path, index=False, encoding="utf-8")
print(f"✅ CSV exporté : {csv_path}")
print(f"✅ Résumé JSON : {SUMMARY_FILE}")
print(f"✅ Données brutes : {RAW_FILE}")
print(f"✅ Passages : {PASSAGES_FILE}")
print(f"✅ Figures : {RESULTS_DIR}/figures/")
print("\nTout est sauvegardé. Rien de perdu 🎉")

## 10. Récapitulatif final

In [ ]:
print(json.dumps(metrics, indent=2, ensure_ascii=False))